<a href="https://colab.research.google.com/github/sukanya9020/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sukanya9020/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

### My baseline rule

I will prioritize content for refresh when it shows a combination of age, update gap, search visibility, and recent performance signals.

The baseline score gives higher priority to content that:
- is older,
- has not been updated recently,
- has meaningful impressions,
- has low CTR,
- and has a negative recent trend.

The score is used only as a decision-support signal for review. It does not claim that refreshing the content will cause better search performance.

### Reason codes

- STALE_LOW_CTR — older content with a long update gap and weak CTR.
- STALE_DECLINE — older content with a long update gap and negative recent trend.
- HIGH_VISIBILITY_LOW_CTR — content receives meaningful impressions but has weak CTR.
- RECENT_DECLINE — content shows a negative recent performance trend.

### Action labels

- Refresh — highest-priority items for human review.
- Review — potentially useful opportunities requiring review.
- Monitor — lower-priority items to watch.
- Protect — relatively healthy items where unnecessary changes should be avoided.

In [ ]:
import pandas as pd
import numpy as np

url = "https://raw.githubusercontent.com/sukanya9020/flyrank-ml-internship/main/data/raw/content_refresh_anonymized.csv"

df = pd.read_csv(url)

print("Rows:", len(df))
print("Columns:", len(df.columns))

df.head()

Rows: 30000
Columns: 44


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


### Signal check 1 — Content age

I will check whether content age is associated with recent search-performance change.

This signal is linked to the refresh/staleness logic discussed in the session. I will compare trend_pct across age buckets and report the observed result without treating it as a causal relationship.

In [ ]:
# Signal 1: Content age buckets

age_bins = [-np.inf, 90, 180, 365, np.inf]
age_labels = ["0-90 days", "91-180 days", "181-365 days", "366+ days"]

df["age_bucket"] = pd.cut(
    df["content_age_days"],
    bins=age_bins,
    labels=age_labels
)

age_check = (
    df.groupby("age_bucket", observed=False)
      .agg(
          n=("content_id", "count"),
          median_trend_pct=("trend_pct", "median"),
          mean_trend_pct=("trend_pct", "mean")
      )
      .reset_index()
)

print("Signal 1 — Content age")
display(age_check)

Signal 1 — Content age


,age_bucket,n,median_trend_pct,mean_trend_pct
0,0-90 days,492,-55.35,5.215801
1,91-180 days,11780,-46.30,-2.317384
2,181-365 days,11368,-32.90,-12.322987
3,366+ days,6360,-14.80,2.106976


**Verdict: OPPOSITE**

The observed bucket results do not support the assumption that older content consistently has a more negative recent trend. The median trend becomes less negative across the older age buckets, from -55.35% for 0–90 days to -14.80% for 366+ days. The mean values also do not show a consistent age-related decline. Therefore, content age should not be treated as a standalone indicator of declining performance. It may still be useful as one component of a broader review score.

In [ ]:
# Signal 2: CTR buckets

ctr_bins = [-np.inf, 0.5, 1.0, 2.0, np.inf]
ctr_labels = ["<0.5%", "0.5-1%", "1-2%", "2%+"]

df["ctr_bucket"] = pd.cut(
    df["ctr"],
    bins=ctr_bins,
    labels=ctr_labels
)

ctr_check = (
    df.groupby("ctr_bucket", observed=False)
      .agg(
          n=("content_id", "count"),
          median_trend_pct=("trend_pct", "median"),
          mean_trend_pct=("trend_pct", "mean")
      )
      .reset_index()
)

print("Signal 2 — CTR")
display(ctr_check)

Signal 2 — CTR


,ctr_bucket,n,median_trend_pct,mean_trend_pct
0,<0.5%,25851,-35.8,-6.998483
1,0.5-1%,2460,-22.3,0.410284
2,1-2%,915,-20.5,6.536719
3,2%+,774,-14.9,39.887602


**Verdict: CONFIRMED**

The observed results show a consistent directional pattern: lower CTR buckets have more negative recent trend values. The median trend improves from -35.8% for CTR below 0.5% to -14.9% for CTR of 2% or higher. The mean trend shows the same general direction. This supports using low CTR as one component of the baseline review score, while avoiding a causal interpretation.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [ ]:
# Section 2 — Build the baseline ranked queue

# Make a working copy
queue = df.copy()

# Convert required fields to numeric values
numeric_cols = [
    "content_age_days",
    "days_since_last_update",
    "impressions_90d",
    "ctr"
]

for col in numeric_cols:
    queue[col] = pd.to_numeric(queue[col], errors="coerce")

# Remove rows that cannot be scored
queue = queue.dropna(subset=numeric_cols).copy()


# -----------------------------
# 1. Create percentile signals
# -----------------------------

# Longer update gap = higher refresh opportunity
update_gap_score = queue["days_since_last_update"].rank(pct=True) * 100

# More impressions = more visible content worth reviewing
visibility_score = queue["impressions_90d"].rank(pct=True) * 100

# Lower CTR = higher review opportunity
low_ctr_score = (1 - queue["ctr"].rank(pct=True)) * 100


# -----------------------------
# 2. Create ONE baseline score
# -----------------------------

queue["baseline_score"] = (
    0.40 * low_ctr_score
    + 0.35 * update_gap_score
    + 0.25 * visibility_score
)


# -----------------------------
# 3. Assign ONE reason code
# -----------------------------

def get_reason_code(row):
    if row["ctr"] < 0.5 and row["days_since_last_update"] >= 90:
        return "STALE_LOW_CTR"
    elif row["ctr"] < 0.5:
        return "LOW_CTR"
    elif row["days_since_last_update"] >= 180:
        return "LONG_UPDATE_GAP"
    else:
        return "BASELINE_REVIEW"


queue["reason_code"] = queue.apply(get_reason_code, axis=1)


# -----------------------------
# 4. Assign action labels
# -----------------------------

def get_action(score):
    if score >= 75:
        return "Refresh"
    elif score >= 50:
        return "Review"
    elif score >= 25:
        return "Monitor"
    else:
        return "Protect"

queue["action"] = queue["baseline_score"].apply(get_action)


# -----------------------------
# 5. Rank the queue
# -----------------------------

queue = queue.sort_values(
    "baseline_score",
    ascending=False
).reset_index(drop=True)

queue["rank"] = queue.index + 1


# -----------------------------
# 6. Select useful output columns
# -----------------------------

output_cols = [
    "rank",
    "content_id",
    "baseline_score",
    "action",
    "reason_code",
    "content_age_days",
    "days_since_last_update",
    "impressions_90d",
    "ctr"
]

baseline_queue = queue[output_cols].copy()


# -----------------------------
# 7. Write the required CSV
# -----------------------------

import os

os.makedirs("work/outputs", exist_ok=True)

output_path = "work/outputs/baseline_action_score.csv"

baseline_queue.to_csv(
    output_path,
    index=False
)

print("Baseline queue created successfully.")
print("Rows scored:", len(baseline_queue))
print("Saved to:", output_path)

display(baseline_queue.head(20))

Baseline queue created successfully.
Rows scored: 30000
Saved to: work/outputs/baseline_action_score.csv


,rank,content_id,baseline_score,action,reason_code,content_age_days,days_since_last_update,impressions_90d,ctr
0,1,content_c65ee459f729,86.652333,Refresh,STALE_LOW_CTR,106,106,6526,0.0
1,2,content_c8e9d6ab9013,85.680000,Refresh,STALE_LOW_CTR,362,104,208678,0.0
2,3,content_b16bd7307b39,85.651250,Refresh,STALE_LOW_CTR,231,194,4590,0.0
3,4,content_fb4bf6555c79,85.512500,Refresh,STALE_LOW_CTR,299,104,84093,0.0
4,5,content_6e28a04c07a8,85.141667,Refresh,STALE_LOW_CTR,139,104,41226,0.0
5,6,content_bc18d49d8f6b,84.906667,Refresh,STALE_LOW_CTR,287,104,32491,0.0
6,7,content_b21385c39124,84.845833,Refresh,STALE_LOW_CTR,286,104,30962,0.0
7,8,content_75175d878762,84.595833,Refresh,STALE_LOW_CTR,299,104,25748,0.0
8,9,content_095661034f9b,84.484167,Refresh,STALE_LOW_CTR,287,104,23513,0.0
9,10,content_825a9788af8d,83.884583,Refresh,STALE_LOW_CTR,421,104,16786,0.0


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [ ]:
# Section 3 — Top-20 review

top20 = baseline_queue.head(20).copy()

def confidence_note(row):
    if row["baseline_score"] >= 75:
        return "High priority because the baseline score is above 75."
    elif row["baseline_score"] >= 50:
        return "Medium priority because the baseline score is between 50 and 75."
    else:
        return "Lower priority because the baseline score is below 50."


def what_would_make_it_wrong(row):
    if row["ctr"] < 0.5 and row["days_since_last_update"] >= 90:
        return "It could be wrong if the low CTR is caused by query mix, tracking issues, or a deliberate low-click search intent."
    elif row["ctr"] < 0.5:
        return "It could be wrong if the low CTR reflects the type of search intent rather than a content problem."
    elif row["days_since_last_update"] >= 180:
        return "It could be wrong if the content remains accurate and useful despite the long update gap."
    else:
        return "It could be wrong if the observed signals do not represent a real content opportunity."


top20_review = top20[
    [
        "rank",
        "content_id",
        "baseline_score",
        "action",
        "reason_code",
        "content_age_days",
        "days_since_last_update",
        "impressions_90d",
        "ctr"
    ]
].copy()

top20_review["confidence_note"] = top20_review.apply(
    confidence_note,
    axis=1
)

top20_review["what_would_make_it_wrong"] = top20_review.apply(
    what_would_make_it_wrong,
    axis=1
)

print("Top-20 review")
display(top20_review)

Top-20 review


,rank,content_id,baseline_score,action,reason_code,content_age_days,days_since_last_update,impressions_90d,ctr,confidence_note,what_would_make_it_wrong
0,1,content_c65ee459f729,86.652333,Refresh,STALE_LOW_CTR,106,106,6526,0.0,High priority because the baseline score is ab...,It could be wrong if the low CTR is caused by ...
1,2,content_c8e9d6ab9013,85.680000,Refresh,STALE_LOW_CTR,362,104,208678,0.0,High priority because the baseline score is ab...,It could be wrong if the low CTR is caused by ...
2,3,content_b16bd7307b39,85.651250,Refresh,STALE_LOW_CTR,231,194,4590,0.0,High priority because the baseline score is ab...,It could be wrong if the low CTR is caused by ...
3,4,content_fb4bf6555c79,85.512500,Refresh,STALE_LOW_CTR,299,104,84093,0.0,High priority because the baseline score is ab...,It could be wrong if the low CTR is caused by ...
4,5,content_6e28a04c07a8,85.141667,Refresh,STALE_LOW_CTR,139,104,41226,0.0,High priority because the baseline score is ab...,It could be wrong if the low CTR is caused by ...
5,6,content_bc18d49d8f6b,84.906667,Refresh,STALE_LOW_CTR,287,104,32491,0.0,High priority because the baseline score is ab...,It could be wrong if the low CTR is caused by ...
6,7,content_b21385c39124,84.845833,Refresh,STALE_LOW_CTR,286,104,30962,0.0,High priority because the baseline score is ab...,It could be wrong if the low CTR is caused by ...
7,8,content_75175d878762,84.595833,Refresh,STALE_LOW_CTR,299,104,25748,0.0,High priority because the baseline score is ab...,It could be wrong if the low CTR is caused by ...
8,9,content_095661034f9b,84.484167,Refresh,STALE_LOW_CTR,287,104,23513,0.0,High priority because the baseline score is ab...,It could be wrong if the low CTR is caused by ...
9,10,content_825a9788af8d,83.884583,Refresh,STALE_LOW_CTR,421,104,16786,0.0,High priority because the baseline score is ab...,It could be wrong if the low CTR is caused by ...


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [ ]:
# Section 4 — Weak picks + leakage check

print("Weak picks / review notes")
print()

# Show the lowest-scoring items in the ranked queue
weak_picks = baseline_queue.tail(10).copy()

display(weak_picks)


# -----------------------------
# Leakage check
# -----------------------------

score_features = [
    "ctr",
    "days_since_last_update",
    "impressions_90d"
]

forbidden_features = [
    "trend_pct",
    "trend_direction"
]

print("Leakage check")
print("----------------")

print("Features used in baseline score:")
for feature in score_features:
    print("-", feature)

print()
print("Outcome/future-related fields excluded from score:")
for feature in forbidden_features:
    print("-", feature)

# Verify that forbidden outcome fields are not part of the scoring features
leakage_found = any(
    feature in score_features
    for feature in forbidden_features
)

if leakage_found:
    print("\nWARNING: Potential leakage detected.")
else:
    print("\nPASS: No trend/future outcome fields were used in the baseline score.")

# Confirm the score columns exist
required_output_columns = [
    "rank",
    "content_id",
    "baseline_score",
    "action",
    "reason_code"
]

missing_columns = [
    col for col in required_output_columns
    if col not in baseline_queue.columns
]

if len(missing_columns) == 0:
    print("PASS: Required ranked-queue columns are present.")
else:
    print("WARNING: Missing columns:", missing_columns)

Weak picks / review notes



,rank,content_id,baseline_score,action,reason_code,content_age_days,days_since_last_update,impressions_90d,ctr
29990,29991,content_a1c65f070bad,3.775333,Protect,BASELINE_REVIEW,116,8,3,33.33
29991,29992,content_dfce82404813,3.775333,Protect,BASELINE_REVIEW,131,8,3,33.33
29992,29993,content_9a7fe374c900,3.695333,Protect,BASELINE_REVIEW,116,8,3,66.67
29993,29994,content_14a42157a627,3.306083,Protect,BASELINE_REVIEW,111,1,8,12.50
29994,29995,content_f26233911f33,3.137750,Protect,BASELINE_REVIEW,140,8,2,50.00
29995,29996,content_b96873ca64c1,3.137750,Protect,BASELINE_REVIEW,144,8,2,50.00
29996,29997,content_4e95a8389562,3.137750,Protect,BASELINE_REVIEW,180,8,2,50.00
29997,29998,content_006b16e7a2e7,2.441833,Protect,BASELINE_REVIEW,140,8,1,100.00
29998,29999,content_4272d3a330a3,2.441833,Protect,BASELINE_REVIEW,144,8,1,100.00
29999,30000,content_cfa4d9f1bf0a,2.441833,Protect,BASELINE_REVIEW,112,8,1,100.00


Leakage check
----------------
Features used in baseline score:
- ctr
- days_since_last_update
- impressions_90d

Outcome/future-related fields excluded from score:
- trend_pct
- trend_direction

PASS: No trend/future outcome fields were used in the baseline score.
PASS: Required ranked-queue columns are present.


### Weak picks and leakage check

The weakest picks are lower-priority items because they have lower combined scores from CTR, update gap, and visibility. These items should not automatically be ignored; they can still be reviewed when there is additional business context.

The baseline score does not use `trend_pct` or `trend_direction`. These fields describe recent performance outcomes and were therefore excluded from the scoring rule to avoid label-derived information entering the baseline.

The baseline also does not use future-window information. The score is based only on currently available content attributes and observed 90-day metrics.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

### Self-check

- [x] Two signal checks were completed with visible bucket tables and sample counts.
- [x] The age signal was marked OPPOSITE based on the observed bucket results.
- [x] The CTR signal was marked CONFIRMED based on the observed bucket results.
- [x] A baseline score was created using currently available content signals.
- [x] One action label was assigned to every scored content item.
- [x] One reason code was assigned to every scored content item.
- [x] The ranked queue was written to `work/outputs/baseline_action_score.csv`.
- [x] A Top-20 review was completed.
- [x] Weak picks were inspected.
- [x] `trend_pct` and `trend_direction` were excluded from the baseline scoring rule.
- [x] No future-window information was used in the baseline score.